# Demo · Fundamentos de NLP

**Tema 2 · Sesión 4 · Máster en Ingeniería de Automatización con IA Agéntica · EBIS**

Este notebook tiene dos usos en clase:

- **Mini-demo en el bloque 2** (slide 9): solo la sección 1 (tokenización con tiktoken). 3 minutos.
- **Demo de cierre** (slide 25): secciones 2, 3 y 4 — embeddings, aritmética semántica y visualización. 10 minutos.

Estructura:

1. **Tokenización** con `tiktoken` — cómo GPT-4 trocea el texto.
2. **Embeddings de frase** con `sentence-transformers` — significado convertido en geometría.
3. **Aritmética semántica** con GloVe — la analogía rey/reina en directo.
4. **Visualización 2D** con PCA — ver los embeddings con los ojos.

> ⏱ Ejecución completa: ~3 min · 🧰 Requisitos: Python 3.10+, MacBook M4 (o cualquier máquina con 4 GB de RAM libres).

## Setup · Instalación y verificación

Instala las dependencias desde `requirements.txt` antes de abrir el notebook y selecciona el kernel de ese mismo entorno. Las celdas no ejecutan instalaciones ad hoc.

In [ ]:
from __future__ import annotations

import os
import platform
import sys

import matplotlib.pyplot as plt
import numpy as np

SENTENCE_MODEL_NAME = "all-MiniLM-L6-v2"
WORD_EMBEDDING_MODEL_NAME = "glove-wiki-gigaword-100"
RANDOM_STATE = 42
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")

print(f"Python  : {sys.version.split()[0]}")
print(f"Sistema : {platform.system()} {platform.release()}")
print(f"CPU     : {platform.machine()}")

In [ ]:
apple_silicon = platform.system() == "Darwin" and platform.machine() == "arm64"
print(f"Apple Silicon: {'sí (MPS opcional)' if apple_silicon else 'no; se usará CPU por defecto'}")
print("Los modelos se descargarán solo si no están disponibles en sus cachés estándar.")

---

# 1 · Tokenización con `tiktoken`

> Esta sección corresponde a la mini-demo de tokenización de las diapositivas.

Una red neuronal opera con números. La tokenización transforma texto en IDs de unidades frecuentes o *subwords*. Un token no equivale necesariamente a una palabra, y BPE, WordPiece y SentencePiece son familias relacionadas pero no idénticas.

In [ ]:
import tiktoken

try:
    enc = tiktoken.encoding_for_model("gpt-4")
    encoding_name = "encoding_for_model('gpt-4')"
except KeyError:
    enc = tiktoken.get_encoding("cl100k_base")
    encoding_name = "cl100k_base (fallback)"

frase = "Hola mundo, soy un LLM."
ids = enc.encode(frase)

assert ids and all(isinstance(token_id, int) for token_id in ids)
print(f"Encoding: {encoding_name}")
print(f"Frase   : {frase!r}")
print(f"Palabras: {len(frase.split())}")
print(f"Tokens  : {len(ids)}")
print(f"IDs     : {ids}")
print("Los IDs concretos dependen del encoding y pueden cambiar al usar otro modelo.")

### Ver los tokens uno a uno

Decodificamos cada ID para ver qué fragmento de texto representa. Observad cómo "Hola" es un token entero pero "LLM" se trocea.

In [ ]:
def mostrar_tokens(texto: str, encoder=enc) -> None:
    token_ids = encoder.encode(texto)
    assert token_ids and all(isinstance(token_id, int) for token_id in token_ids)
    print(f"\n{texto!r}\n→ {len(token_ids)} tokens:")
    for indice, token_id in enumerate(token_ids):
        token_bytes = encoder.decode_single_token_bytes(token_id)
        try:
            fragmento = token_bytes.decode("utf-8")
        except UnicodeDecodeError:
            fragmento = repr(token_bytes)
        print(f"  [{indice:2d}] ID={token_id:6d} → {fragmento!r}")


mostrar_tokens("Hola mundo, soy un LLM.")

### Comparativa español/inglés

Comparamos ejemplos breves en ambos idiomas. La relación palabras/tokens depende del texto y del encoding: no existe un porcentaje universal de diferencia de coste.

In [ ]:
ejemplos_idioma = [
    ("ES", "El procesamiento del lenguaje natural es fascinante."),
    ("EN", "Natural language processing is fascinating."),
    ("ES", "Esternocleidomastoideo."),
    ("EN", "Antidisestablishmentarianism."),
    ("ES", "¡Hola! 👋 ¿Cómo estás? 😀"),
    ("ES", "¡Muchas gracias!"),
    ("EN", "Thanks a lot!"),
]

for idioma, texto in ejemplos_idioma:
    n_tokens = len(enc.encode(texto))
    n_palabras = len(texto.split())
    ratio = n_tokens / n_palabras
    print(
        f"[{idioma}] {texto!r:60} → {n_tokens:2d} tokens "
        f"({n_palabras} palabras, {ratio:.2f} tokens/palabra)"
    )

### Casos divertidos · palabras técnicas, emojis y código

Cualquier texto del mundo es representable, pero la eficiencia varía.

In [ ]:
casos = [
    "tokenización",
    "Schnauzer",
    "🦄🌈✨",
    "def hello():\n    print('hi')",
    "0xDEADBEEF",
]

for caso in casos:
    mostrar_tokens(caso)

> **🛑 Pausa de mini-demo aquí.** Si estás en el slide 9 de la sesión, vuelve a las slides. Continúa con el notebook cuando llegues al slide 25.

---

# 2 · Embeddings de frase con `sentence-transformers`

Usamos `all-MiniLM-L6-v2` para convertir frases completas en vectores. La primera carga requiere red; las siguientes reutilizan la caché de Hugging Face.

In [ ]:
from sentence_transformers import SentenceTransformer

try:
    print(f"Cargando {SENTENCE_MODEL_NAME!r}...")
    model = SentenceTransformer(SENTENCE_MODEL_NAME, device="cpu")
except (OSError, RuntimeError) as exc:
    raise RuntimeError(
        "No se pudo cargar el modelo de frases. Comprueba red, proxy o certificado; "
        "si trabajarás offline, precárgalo una vez con conexión."
    ) from exc

embedding_dimension = model.get_embedding_dimension()
print(f"Modelo listo. Dimensión de salida: {embedding_dimension}")

### Embeddings de varias frases

Codificamos seis frases. Tres sobre tecnología, tres sobre cocina. Veremos en el siguiente paso que el espacio las separa solo.

In [ ]:
frases_embedding = [
    "Los Transformers revolucionaron el procesamiento del lenguaje natural.",
    "BERT y GPT se basan en la arquitectura Transformer.",
    "La inteligencia artificial generativa transforma el desarrollo de software.",
    "Una buena tortilla de patatas necesita cebolla.",
    "El secreto de la paella es el sofrito y el caldo.",
    "Cocinar pasta al dente requiere práctica.",
]

embeddings = np.asarray(
    model.encode(frases_embedding, normalize_embeddings=True),
    dtype=np.float32,
)

assert embeddings.shape == (len(frases_embedding), embedding_dimension)
assert np.isfinite(embeddings).all()
print(f"Forma del array: {embeddings.shape}")
print(f"Primeras 8 dimensiones:\n{embeddings[0, :8]}")

### Matriz de similitud coseno

Calculamos la similitud entre cada par de frases. Los embeddings están normalizados, así que el producto escalar **es** la similitud coseno.

Esperamos:
- Alta similitud dentro de cada grupo temático (tecnología y cocina).
- Baja similitud entre grupos.

In [ ]:
similitud = embeddings @ embeddings.T
n_frases = len(frases_embedding)

assert similitud.shape == (n_frases, n_frases)
assert np.isfinite(similitud).all()
assert np.allclose(similitud, similitud.T, atol=1e-6)
assert np.allclose(np.diag(similitud), 1.0, atol=1e-5)

print("Matriz de similitud coseno:\n")
print(" " * 4 + "  ".join(f"F{i}" for i in range(n_frases)))
for i, fila in enumerate(similitud):
    print(f"F{i}  " + "  ".join(f"{valor:+.2f}" for valor in fila))

print("\nLeyenda:")
for i, texto in enumerate(frases_embedding):
    print(f"  F{i}: {texto}")

### Visualización como heatmap

Más visual: las celdas oscuras son frases similares; las claras, distintas. Verás dos cuadrados oscuros en la diagonal (tecnología arriba-izquierda, cocina abajo-derecha) y celdas claras en los cuadrantes cruzados.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(similitud, cmap="viridis", vmin=-1, vmax=1)
ax.set_xticks(range(n_frases), labels=[f"F{i}" for i in range(n_frases)])
ax.set_yticks(range(n_frases), labels=[f"F{i}" for i in range(n_frases)])

for i in range(n_frases):
    for j in range(n_frases):
        valor = similitud[i, j]
        ax.text(
            j,
            i,
            f"{valor:.2f}",
            ha="center",
            va="center",
            color="white" if valor < 0.45 else "black",
            fontsize=9,
        )

plt.colorbar(im, ax=ax, label="Similitud coseno")
ax.set_title("Similitud semántica entre frases")
plt.tight_layout()
plt.show()

### Interpretación de los extremos

Se excluye la diagonal para identificar el par más parecido y el menos parecido. La similitud orienta una comparación semántica, pero no constituye por sí sola una decisión o clasificación.

In [ ]:
pares = [
    (i, j, float(similitud[i, j]))
    for i in range(n_frases)
    for j in range(i + 1, n_frases)
]
mas_similar = max(pares, key=lambda item: item[2])
menos_similar = min(pares, key=lambda item: item[2])

for etiqueta, (i, j, valor) in [
    ("Par más similar", mas_similar),
    ("Par menos similar", menos_similar),
]:
    print(f"{etiqueta}: F{i} / F{j} = {valor:.3f}")
    print(f"  {frases_embedding[i]}")
    print(f"  {frases_embedding[j]}\n")

---

# 3 · Aritmética semántica con GloVe

GloVe representa palabras inglesas individuales. Las analogías son aproximaciones dependientes del corpus: pueden producir vecinos plausibles sin devolver exactamente el término esperado.

In [ ]:
import gensim.downloader as api

try:
    print(f"Cargando {WORD_EMBEDDING_MODEL_NAME!r}...")
    glove = api.load(WORD_EMBEDDING_MODEL_NAME)
except (OSError, RuntimeError, ValueError) as exc:
    raise RuntimeError(
        "No se pudo cargar GloVe. Comprueba red, proxy y espacio disponible; "
        "para trabajar offline, precarga el modelo una vez con conexión."
    ) from exc

print(
    f"GloVe listo. Vocabulario: {len(glove.key_to_index):,}; "
    f"dimensión: {glove.vector_size}"
)

In [ ]:
def resolver_analogia(
    positivos: list[str],
    negativos: list[str],
    topn: int = 5,
) -> list[tuple[str, float]]:
    entradas = set(positivos + negativos)
    ausentes = sorted(palabra for palabra in entradas if palabra not in glove.key_to_index)
    if ausentes:
        print(f"No se puede calcular: fuera de vocabulario: {ausentes}")
        return []

    candidatos = glove.most_similar(
        positive=positivos,
        negative=negativos,
        topn=topn + len(entradas),
    )
    return [(palabra, score) for palabra, score in candidatos if palabra not in entradas][:topn]


resultado_realeza = resolver_analogia(["king", "woman"], ["man"])
print("king - man + woman ≈ ?\n")
for palabra, score in resultado_realeza:
    print(f"  {palabra:15s} (similitud: {score:.4f})")

### Más analogías

Se muestran varios vecinos, sin seleccionar manualmente el resultado. La analogía de capitales sirve como segundo ejemplo clásico; plural y tiempo verbal ilustran que algunas relaciones son menos estables.

In [ ]:
analogias = [
    ("Capital", ["paris", "italy"], ["france"], "rome"),
    ("Plural", ["cats", "dog"], ["cat"], "dogs"),
    ("Pasado", ["walked", "swim"], ["walk"], "swam"),
    ("Alternativa", ["berlin", "france"], ["germany"], "paris"),
]

for etiqueta, positivos, negativos, esperado in analogias:
    vecinos = resolver_analogia(positivos, negativos, topn=5)
    formula = " + ".join(positivos) + "".join(f" - {p}" for p in negativos)
    print(f"\n[{etiqueta}] {formula} ≈ ? (referencia esperada: {esperado})")
    if vecinos:
        print("  " + ", ".join(f"{p} ({s:.3f})" for p, s in vecinos))
        posicion = next((i + 1 for i, (p, _) in enumerate(vecinos) if p == esperado), None)
        print(f"  Resultado esperado en top 5: {posicion if posicion else 'no'}")

> **Interpretación:** las regularidades geométricas pueden capturar relaciones semánticas y sintácticas, pero no son reglas lógicas ni garantías. El resultado depende del corpus, el modelo y el vocabulario.

---

# 4 · Visualización 2D con PCA

Los embeddings tienen 384 dimensiones (MiniLM) o 100 (GloVe). No podemos visualizarlos directamente. Los **proyectamos a 2D** con PCA, perdiendo información pero ganando intuición.

In [ ]:
from sklearn.decomposition import PCA

grupos_palabras = {
    "Animales": ["cat", "dog", "lion", "tiger", "elephant", "horse"],
    "Frutas": ["apple", "banana", "orange", "grape", "strawberry", "pineapple"],
    "Tecnología": ["computer", "software", "internet", "algorithm", "data", "network"],
}

registros = [
    (palabra, categoria)
    for categoria, palabras_categoria in grupos_palabras.items()
    for palabra in palabras_categoria
    if palabra in glove.key_to_index
]
ausentes = [
    palabra
    for palabras_categoria in grupos_palabras.values()
    for palabra in palabras_categoria
    if palabra not in glove.key_to_index
]
if ausentes:
    print(f"Palabras omitidas por estar fuera del vocabulario: {ausentes}")

palabras = [palabra for palabra, _ in registros]
categorias = [categoria for _, categoria in registros]
vectores = np.asarray([glove[palabra] for palabra in palabras], dtype=np.float32)

assert len(palabras) >= 3
assert vectores.shape == (len(palabras), glove.vector_size)
assert np.isfinite(vectores).all()

pca = PCA(n_components=2, random_state=RANDOM_STATE)
vectores_2d = pca.fit_transform(vectores)
assert vectores_2d.shape == (len(palabras), 2)

print(f"Vectores originales: {vectores.shape}")
print(f"Proyección PCA: {vectores_2d.shape}")
print(f"Varianza explicada en 2D: {pca.explained_variance_ratio_.sum() * 100:.1f}%")

In [ ]:
colores = {"Animales": "#08627F", "Frutas": "#6ACBB8", "Tecnología": "#C46A1A"}

fig, ax = plt.subplots(figsize=(11, 7))
for categoria in grupos_palabras:
    indices = [i for i, valor in enumerate(categorias) if valor == categoria]
    if not indices:
        continue
    ax.scatter(
        vectores_2d[indices, 0],
        vectores_2d[indices, 1],
        c=colores[categoria],
        label=categoria,
        s=110,
        alpha=0.82,
        edgecolors="white",
        linewidths=1.2,
    )

for i, palabra in enumerate(palabras):
    desplazamiento_y = 8 if i % 2 == 0 else -14
    ax.annotate(
        palabra,
        (vectores_2d[i, 0], vectores_2d[i, 1]),
        xytext=(6, desplazamiento_y),
        textcoords="offset points",
        fontsize=9,
    )

ax.set_xlabel("Componente principal 1")
ax.set_ylabel("Componente principal 2")
ax.set_title("Embeddings GloVe proyectados a 2D con PCA")
ax.legend()
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

> PCA comprime 100 dimensiones en solo dos y necesariamente pierde información. La cercanía visual permite explorar patrones, pero no reproduce exactamente las distancias del espacio original ni prueba por sí sola la existencia de categorías.

---

# Cierre · Lo que hemos visto

- **Tokenización:** el texto se transforma en IDs de tokens o *subwords*.
- **Embeddings:** palabras y frases se representan mediante vectores.
- **Similitud coseno:** permite comparar orientación semántica entre vectores.
- **PCA:** ofrece una vista exploratoria 2D con pérdida de información.

Las diapositivas conectan estos fundamentos con RNN, LSTM, atención y Transformers. En esas arquitecturas, la secuencia continúa conceptualmente así:

```text
Texto → tokenización → IDs → embeddings → atención/Transformer → comprensión o generación
```

Este notebook no implementa ni entrena esos modelos: mantiene una demo breve, local y reproducible de los fundamentos.